# ColabUMA Opt — UMAによる分子構造最適化

Metaの機械学習ポテンシャル **UMA** で分子の構造最適化を行い、最適化した構造で **DFT の一点計算**（[GPU4PySCF](https://github.com/pyscf/gpu4pyscf)）を行うノートブックです。コードを書く必要はありません。

## 使い方

1. メニューの **ランタイム → ランタイムのタイプを変更** で **T4 GPU** を選ぶ
2. 初回のみ、左サイドバーの🔑（シークレット）に名前 `HF_TOKEN`、値にHugging Faceのアクセストークンを登録し、「ノートブックからのアクセス」をオンにする
3. 下の **1. Input form** で出力名、電荷、スピン多重度、収束判定、DFT の条件を指定する
4. **ランタイム → すべて実行** を押す
5. **2. File selection and checks** にファイル選択ボタンが出るので、xyz または Gaussian の gjf / com を選ぶ。あとは放置できます
6. 下のセルで 3D 表示・エネルギー・収束グラフ・DFT のエネルギーを確認し、最後のセルで結果のZIPをダウンロードする

gjfからは構造だけを読み込みます。電荷と多重度はフォームの値を使います。ZIPの中の `_opt.log` は最適化の経路をGaussianの出力形式で書いたもので、GaussViewで開くと各ステップの構造とエネルギー変化を確認できます（Gaussianで計算した結果ではありません）。DFT のエネルギーは `_dft.csv` に入ります。

2回目以降は手順3〜6を繰り返すだけです（インストールとモデル読み込みは自動でスキップされます）。ファイル選択で Cancel upload を押すと、前回のファイルをそのまま使います。

> **事前準備**：モデルの重み [facebook/UMA](https://huggingface.co/facebook/UMA) はアクセス申請が必要です。Hugging Faceにログインしてページ上でFAIR Chemistry License v1に同意し、承認後に [Access Tokens](https://huggingface.co/settings/tokens) で Read 権限のトークンを作成してください。

## 注意事項

- 最適化は鞍点に収束することがあります。重要な構造は **振動計算** をオンにして確認してください。
- UMAは機械学習ポテンシャルであり、DFTの代わりにはなりません。最終的なエネルギー比較はDFTで確認してください。
- 小さな分子（数十原子以下）では、DFTを直接使う方が確実な場面が多くあります。
- DFT の一点計算は UMA の構造のままで、DFT で構造を最適化し直すものではありません。開殻は UKS で計算します。
- DFT は GPU4PySCF（密度フィッティング）で計算します。T4 は倍精度の計算が遅いため、原子数が多いときや def2-TZVPD では時間がかかります。GPU が使えないときは CPU の PySCF で計算します。
- 周期系（結晶・表面）の最適化には対応していません。
- 結合長・角度・二面角のスキャンは [ColabUMA_Scan](https://colab.research.google.com/github/RChibaChiba/ColabChem/blob/main/ColabUMA_Scan.ipynb)、遷移状態の探索は [ColabUMA_TS](https://colab.research.google.com/github/RChibaChiba/ColabChem/blob/main/ColabUMA_TS.ipynb) を使ってください。最適化 → スキャン → TS を同じセッションで続けるときは [ColabUMA_Workflow](https://colab.research.google.com/github/RChibaChiba/ColabChem/blob/main/ColabUMA_Workflow.ipynb) を使うと、インストールとモデル読み込みが1回で済みます。

## ライセンス

- このノートと共通コード（`colabuma`）：[MIT License](https://github.com/RChibaChiba/ColabChem/blob/main/LICENSE)
- UMAのモデル重み：[FAIR Chemistry License v1](https://huggingface.co/facebook/UMA) に従います
- 実行時にインストールするライブラリは、それぞれのライセンスに従います。一覧は [README のライセンス](https://github.com/RChibaChiba/ColabChem/blob/main/README.ja.md#ライセンス) にあります

## 引用

論文で使う場合は、以下を引用してください。

- B. M. Wood *et al.*, "UMA: A Family of Universal Models for Atoms", arXiv:2506.23971 (2025).
- D. S. Levine *et al.*, "The Open Molecules 2025 (OMol25) Dataset, Evaluations, and Models", arXiv:2505.08762 (2025).
- fairchem: https://github.com/facebookresearch/fairchem

DFT の一点計算を使った場合は、次も引用してください。

- PySCF（Q. Sun *et al.*, *J. Chem. Phys.* **153**, 024109 (2020)）
- GPU4PySCF（X. Wu *et al.*, arXiv:2404.09452 (2024)）

In [ ]:
#@title 1. Input form { display-mode: "form" }
#@markdown When the form is filled in, choose **Runtime → Run all**. Pick an xyz or gjf file in the next cell; after that you can leave it running.

#@markdown ### Output
output_name = "result"  #@param {type:"string"}

#@markdown ### Structure
charge = 0  #@param {type:"integer"}
multiplicity = 1  #@param {type:"integer"}

#@markdown ### Settings
model_name = "uma-s-1p1"  #@param ["uma-s-1p1", "uma-s-1p2", "uma-m-1p1"]
task_name = "omol"  #@param ["omol"]
optimizer_name = "LBFGS"  #@param ["LBFGS", "BFGS", "FIRE"]
#@markdown Convergence threshold fmax (eV/Å). normal is close to Gaussian's default, tight to Gaussian's opt=tight.
convergence = "loose (0.05)"  #@param ["loose (0.05)", "normal (0.02)", "tight (0.001)", "custom"]
#@markdown fmax (eV/Å) used when custom is chosen.
fmax_custom = 0.05  #@param {type:"number"}
max_steps = 500  #@param {type:"integer"}
run_vibrations = False  #@param {type:"boolean"}

#@markdown ### DFT single point (gpu4pyscf)
#@markdown Computes the DFT energy at the optimized structure. ωB97M-V/def2-TZVPD is the level of theory of UMA's training data (OMol25). def2-TZVPD can take tens of minutes per point on a T4.
run_dft = False  #@param {type:"boolean"}
dft_functional = "B3LYP-D3(BJ)"  #@param ["ωB97M-V (same as OMol25)", "B3LYP-D3(BJ)", "PBE0-D3(BJ)", "M06-2X", "r2SCAN-D4"]
dft_basis = "def2-SVP"  #@param ["def2-SVP", "def2-TZVP", "def2-TZVPD"]
#@markdown Time limit for DFT (minutes, all structures together). Past it, DFT is skipped and the UMA results are saved. 0 means no limit (DFT still stops before the Colab runtime is likely to end).
dft_time_limit = 60  #@param {type:"integer"}

In [ ]:
#@title 2. File selection and checks { display-mode: "form" }
#@markdown File selection and the token check happen here. You can leave the install and calculations after this running.

import os
import subprocess
import sys

COLABUMA_REF = "main"  # GitHub branch or tag of the shared code (colabuma package)


def _use_colabuma(ref):
    # Use colabuma next to the notebook if it is there, otherwise fetch it from GitHub
    if os.path.isdir("colabuma"):
        path = os.getcwd()
    else:
        path = "/content/ColabChem" if os.path.isdir("/content") else os.path.expanduser("~/ColabChem")
        if not os.path.isdir(os.path.join(path, "colabuma")):
            r = subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", ref,
                                "https://github.com/RChibaChiba/ColabChem.git", path], capture_output=True, text=True)
            if r.returncode != 0 or not os.path.isdir(os.path.join(path, "colabuma")):
                raise RuntimeError(f"Could not fetch the ColabChem shared code from GitHub (ref {ref}).\n{r.stderr}")
    if path not in sys.path:
        sys.path.insert(0, path)


_use_colabuma(COLABUMA_REF)

from colabuma import dft, env, opt, output, structure, ui
from colabuma.ui import ColabUMAError, ok

ui.install_error_handler()

fmax = opt.fmax_from_form(convergence, fmax_custom)
structure.check_form(fmax, max_steps, multiplicity)
job = opt.Job(structure.out_prefix(output_name), charge, multiplicity, model_name, task_name,
              optimizer_name, fmax, max_steps, convergence)
if run_dft:
    dft.parse_method(dft_functional, dft_basis)
    if dft_time_limit < 0:
        raise ColabUMAError("The DFT time limit must be 0 or more.", "0 means no limit.",
                            ja="DFT の時間の上限は0以上にしてください。", ja_hint="0 なら上限なしです。")
env.check_gpu()
env.ensure_token()
upload_name, upload_data = structure.upload()
ok("Input checked. You can leave it running from here.")

In [ ]:
#@title 3. Setup (install and login) { display-mode: "form" }
DEVICE = env.setup(["dft"] if run_dft else [])

In [ ]:
#@title 4. Load the model { display-mode: "form" }
calc = env.load_calculator(model_name, task_name)

In [ ]:
#@title 5. Prepare the structures { display-mode: "form" }
atoms0, source_desc = structure.read_structure(upload_name, upload_data, charge, multiplicity)
ok(f"Initial structure ready: {atoms0.get_chemical_formula()} ({len(atoms0)} atoms).")
if run_dft:
    dft.check_memory([("", atoms0)], charge, multiplicity, dft.parse_method(dft_functional, dft_basis)[1])

In [ ]:
#@title 6. Optimization { display-mode: "form" }
#@markdown The Converge bar shows how far fmax has come, in orders of magnitude, from its first value toward the threshold. It moves back if fmax rises.
print(f"Optimizing with {optimizer_name} (fmax < {fmax} eV/Å, at most {max_steps} steps)")
best = opt.optimize(atoms0, calc, job, "Structure")
vib_result = opt.vibrations(best["atoms"], calc) if run_vibrations else None

In [ ]:
#@title 7. Results { display-mode: "form" }
files, log = opt.report(best, vib_result, job)

In [ ]:
#@title 8. DFT single points { display-mode: "form" }
dft_log = []
if run_dft:
    _f, dft_log = dft.run_optional([("Optimized structure", best["atoms"])], charge, multiplicity, dft_functional, dft_basis,
                                   None, dft_time_limit)
    files.update(_f)
else:
    print("DFT single points are off.")

In [ ]:
#@title 9. Save { display-mode: "form" }
files["log.txt"] = output.log_text("ColabUMA Opt log", source_desc, job, [log, dft_log])
zip_path = output.save_zip(job.out_prefix, files)